In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam, Nadam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import metrics
import ray
from ray import tune

# ========== CARREGAMENTO E PREPARAÇÃO DOS DADOS ==========
# Carregar o arquivo (substitua pelo seu caminho)
df = pd.read('dados.txt')  # ou df = pd.read_excel('seu_arquivo.xlsx')
df = df[1000:3001000]
# Normalizar os dados
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

# ========== DEFINIÇÕES DE PARÂMETROS ==========
# Definir parâmetros (substitua pelos seus valores)
timestepsValue = 80  # Número de timesteps
n_timesteps = timestepsValue
n_features = df_scaled.shape[1]  # Número de colunas (X, Y, Z)

# Configurar TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=3, test_size=2, gap=2)

# Pasta para salvar modelos
model_folder = 'modelos_lstm'
os.makedirs(model_folder, exist_ok=True)

# ========== FUNÇÃO PARA CRIAR SEQUÊNCIAS ==========
def create_sequences(data, n_timesteps):
    X, y = [], []
    for i in range(len(data) - n_timesteps):
        X.append(data[i:i+n_timesteps, :])  # Sequência de entrada
        y.append(data[i+n_timesteps, :])   # Próxima linha como saída
    return np.array(X), np.array(y)

# ========== PREPARAR DADOS PARA OTIMIZAÇÃO ==========
# Usar apenas o primeiro split para otimização de hiperparâmetros
train_index, test_index = next(tscv.split(df_scaled.values))
train_data, test_data = df_scaled.values[train_index], df_scaled.values[test_index]
X_train, y_train = create_sequences(train_data, n_timesteps)
X_test, y_test = create_sequences(test_data, n_timesteps)

print(f"Dados preparados para otimização:")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# ========== FUNÇÃO DE OTIMIZAÇÃO ==========
def train_lstm(config):
    # Usar os dados já preparados
    X_local, y_local = X_train, y_train
    
    model = Sequential()
    model.add(Input(shape=(n_timesteps, n_features)))
    model.add(LSTM(
        units=int(config['units']),
        activation=config['activation'],
        return_sequences=False
    ))
    model.add(Dropout(config['dropout']))
    model.add(Dense(n_features))  # Saída com mesmo número de features

    # Escolher otimizador
    if config['optimizer'] == 'Adam':
        optimizer = Adam(learning_rate=config['lr'])
    elif config['optimizer'] == 'Nadam':
        optimizer = Nadam(learning_rate=config['lr'])
    else:
        optimizer = RMSprop(learning_rate=config['lr'])

    model.compile(optimizer=optimizer, loss='mse')

    history = model.fit(
        X_local, y_local,
        epochs=config['epochs'],
        batch_size=config['batch'],
        validation_split=0.2,
        callbacks=[EarlyStopping(monitor='val_loss', patience=config['patience'], mode='min')],
        verbose=0
    )

    val_loss = history.history['val_loss'][-1]
    return {'loss': val_loss}

# ========== ESPAÇO DE BUSCA E OTIMIZAÇÃO ==========
search_space = {
    'units': tune.choice([64, 96, 128, 160, 192, 224, 256, 330]),
    'lr': tune.loguniform(1e-5, 1e-2),
    'dropout': tune.uniform(0.0, 0.5),
    'batch': tune.choice([64, 96, 124, 150, 200]),
    'activation': tune.choice(['tanh', 'relu']),
    'optimizer': tune.choice(['Adam', 'Nadam', 'RMSprop']),
    'epochs': tune.choice([30, 50, 75]),
    'patience': tune.choice([5, 10, 15])
}

# Executar a busca
print("Iniciando otimização de hiperparâmetros...")
analysis = tune.run(
    train_lstm,
    config=search_space,
    num_samples=25,
    metric='loss',
    mode='min',
    time_budget_s=900  # Tempo total em segundos
)

# Melhor configuração
melhor_config = analysis.best_config
print("Melhor configuração encontrada:")
for chave, valor in melhor_config.items():
    print(f"{chave}: {valor}")

# Salvar os melhores hiperparâmetros
best_lr = melhor_config['lr']
neuroniosValue = melhor_config['units']
ativacaoType = melhor_config['activation']
dropoutValue = melhor_config['dropout']
batchValue = melhor_config['batch']
epocasValue = melhor_config['epochs']
pacienciaValue = melhor_config['patience']
otimizadorType = melhor_config['optimizer']

: 